<a href="https://colab.research.google.com/github/sittana-afifi/KTT-Fellow-Hackathon---G1---T2.1-Compressed-Crop-Disease-Classifier/blob/main/train_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import os
import shutil

# --- 1. CLEANUP & PATHS ---
DATA_DIR = '/content/ktt/data/mini_plant_set'
MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Remove hidden Colab folders that cause the "FileNotFoundError"
checkpoint_path = os.path.join(DATA_DIR, '.ipynb_checkpoints')
if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)
    print("🧹 Cleaned hidden checkpoint folders.")

# --- 2. DATA PREPARATION ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load dataset (Filtering out any non-directory files)
full_dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
class_names = full_dataset.classes

# 80% Train, 20% Test
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_data, test_data = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

print(f"✅ Ready! Training on {train_size} images across {len(class_names)} classes.")

# --- 3. MODEL SETUP (MobileNetV3-Small) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')

# Freeze base, replace head
for param in model.parameters():
    param.requires_grad = False

num_ftrs = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

# --- 4. TRAINING ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier[3].parameters(), lr=0.001)

print(f"🚀 Training on {device}...")
for epoch in range(10):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/10 | Loss: {total_loss/len(train_loader):.4f}")

# --- 5. EXPORT ---
# Save standard weights
torch.save(model.state_dict(), f"{MODEL_DIR}/hinga_model.pth")

# Export to ONNX for the API
dummy_input = torch.randn(1, 3, 224, 224).to(device)
torch.onnx.export(model, dummy_input, f"{MODEL_DIR}/model.onnx",
                  input_names=['input'], output_names=['output'])

print(f"📦 Model exported to {MODEL_DIR}/model.onnx")

✅ Ready! Training on 1200 images across 5 classes.
🚀 Training on cpu...
Epoch 1/10 | Loss: 0.6073
Epoch 2/10 | Loss: 0.1325
Epoch 3/10 | Loss: 0.0858
Epoch 4/10 | Loss: 0.0657
Epoch 5/10 | Loss: 0.0530
Epoch 6/10 | Loss: 0.0446
Epoch 7/10 | Loss: 0.0381
Epoch 8/10 | Loss: 0.0357
Epoch 9/10 | Loss: 0.0345
Epoch 10/10 | Loss: 0.0266


/tmp/ipykernel_463/1149087118.py:78: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(model, dummy_input, f"{MODEL_DIR}/model.onnx",
W0423 11:26:05.329000 463 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0423 11:26:05.331000 463 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0423 11:26:05.334000 463 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_

[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
📦 Model exported to models/model.onnx
